In [0]:

from pyspark.sql.functions import (
    monotonically_increasing_id,
    current_timestamp,
    current_date,
    to_date,
    col,
    lit,
    trim
)
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "workspace")
catalog_name = dbutils.widgets.get("catalog_name")

print("=" * 50)
print(f"Catalog: {catalog_name}")
print("=" * 50)

In [0]:
dim_artist = spark.table(f"{catalog_name}.silver.artist") \
    .withColumnRenamed("ArtistId", "artist_id") \
    .withColumnRenamed("Name",     "artist_name") \
    .withColumn("artist_key",   monotonically_increasing_id()) \
    .withColumn("created_date", current_timestamp()) \
    .select("artist_key", "artist_id", "artist_name", "created_date")

dim_artist.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_artist")

print(f"✅ dim_artist: {spark.table(f'{catalog_name}.gold.dim_artist').count()} rows")

In [0]:
dim_album = spark.table(f"{catalog_name}.silver.album") \
    .withColumnRenamed("AlbumId",  "album_id") \
    .withColumnRenamed("Title",    "album_title") \
    .withColumnRenamed("ArtistId", "artist_id") \
    .withColumn("album_key",    monotonically_increasing_id()) \
    .withColumn("created_date", current_timestamp()) \
    .select("album_key", "album_id", "album_title", "artist_id", "created_date")

dim_album.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_album")

print(f"✅ dim_album: {spark.table(f'{catalog_name}.gold.dim_album').count()} rows")

In [0]:
dim_genre = spark.table(f"{catalog_name}.silver.genre") \
    .withColumnRenamed("GenreId", "genre_id") \
    .withColumnRenamed("Name",    "genre_name") \
    .withColumn("genre_key",    monotonically_increasing_id()) \
    .withColumn("created_date", current_timestamp()) \
    .select("genre_key", "genre_id", "genre_name", "created_date")

dim_genre.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_genre")

print(f"✅ dim_genre: {spark.table(f'{catalog_name}.gold.dim_genre').count()} rows")

In [0]:
dim_media_type = spark.table(f"{catalog_name}.silver.mediatype") \
    .withColumnRenamed("MediaTypeId", "media_type_id") \
    .withColumnRenamed("Name",        "media_type_name") \
    .withColumn("media_type_key", monotonically_increasing_id()) \
    .withColumn("created_date",   current_timestamp()) \
    .select("media_type_key", "media_type_id", "media_type_name", "created_date")

dim_media_type.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_media_type")

print(f"✅ dim_media_type: {spark.table(f'{catalog_name}.gold.dim_media_type').count()} rows")

In [0]:
dim_employee = spark.table(f"{catalog_name}.silver.employee") \
    .withColumnRenamed("EmployeeId", "employee_id") \
    .withColumnRenamed("FirstName",  "first_name") \
    .withColumnRenamed("LastName",   "last_name") \
    .withColumnRenamed("Title",      "title") \
    .withColumnRenamed("ReportsTo",  "reports_to") \
    .withColumnRenamed("City",       "city") \
    .withColumnRenamed("Country",    "country") \
    .withColumn("birth_date",    to_date(col("BirthDate"))) \
    .withColumn("hire_date",     to_date(col("HireDate"))) \
    .withColumn("employee_key",  monotonically_increasing_id()) \
    .withColumn("created_date",  current_timestamp()) \
    .select("employee_key", "employee_id", "first_name", "last_name",
            "title", "reports_to", "birth_date", "hire_date",
            "city", "country", "created_date")

dim_employee.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_employee")

print(f"✅ dim_employee: {spark.table(f'{catalog_name}.gold.dim_employee').count()} rows")

In [0]:
dim_track = spark.table(f"{catalog_name}.silver.track") \
    .withColumnRenamed("TrackId",      "track_id") \
    .withColumnRenamed("Name",         "track_name") \
    .withColumnRenamed("AlbumId",      "album_id") \
    .withColumnRenamed("MediaTypeId",  "media_type_id") \
    .withColumnRenamed("GenreId",      "genre_id") \
    .withColumnRenamed("Composer",     "composer") \
    .withColumnRenamed("Milliseconds", "milliseconds") \
    .withColumnRenamed("UnitPrice",    "unit_price") \
    .withColumn("track_key",    monotonically_increasing_id()) \
    .withColumn("created_date", current_timestamp()) \
    .select("track_key", "track_id", "track_name", "album_id",
            "media_type_id", "genre_id", "composer",
            "milliseconds", "unit_price", "created_date")

dim_track.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_track")

print(f"✅ dim_track: {spark.table(f'{catalog_name}.gold.dim_track').count()} rows")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.gold.dim_customer")
print("✅ Dropped existing dim_customer")

In [0]:
result = spark.sql(f"SHOW TABLES IN {catalog_name}.gold").toPandas()
print(result)

In [0]:
# Drop just in case any metadata ghost exists
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.gold.dim_customer")

new_customers = spark.table(f"{catalog_name}.silver.customer") \
    .withColumnRenamed("CustomerId", "customer_id") \
    .withColumnRenamed("FirstName",  "first_name") \
    .withColumnRenamed("LastName",   "last_name") \
    .withColumnRenamed("Email",      "email") \
    .withColumnRenamed("City",       "city") \
    .withColumnRenamed("Country",    "country") \
    .select("customer_id", "first_name", "last_name", "email", "city", "country")

init_df = new_customers \
    .withColumn("customer_key",         monotonically_increasing_id()) \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date",   lit(None).cast("date")) \
    .withColumn("is_current",           lit(True))

init_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.dim_customer")

print(f"✅ dim_customer initial load: {spark.table(f'{catalog_name}.gold.dim_customer').count()} rows")

spark.table(f"{catalog_name}.gold.dim_customer") \
    .select("customer_id", "email", "city", "effective_start_date", "effective_end_date", "is_current") \
    .orderBy("customer_id") \
    .show(5)

In [0]:
from pyspark.sql.functions import countDistinct, sum as spark_sum, max as spark_max, min as spark_min, to_date, col

fact_sales = spark.table(f"{catalog_name}.silver.invoiceline").alias("il") \
    .join(
        spark.table(f"{catalog_name}.silver.invoice").alias("i"),
        col("il.InvoiceId") == col("i.InvoiceId")
    ) \
    .join(
        spark.table(f"{catalog_name}.gold.dim_customer")
             .filter(col("is_current") == True).alias("dc"),
        col("i.CustomerId") == col("dc.customer_id")
    ) \
    .select(
        col("il.InvoiceLineId").alias("invoice_line_key"),
        col("i.InvoiceId").alias("invoice_id"),
        col("dc.customer_key"),
        to_date(col("i.InvoiceDate")).alias("invoice_date"),
        col("il.TrackId").alias("track_id"),
        col("il.UnitPrice").alias("unit_price"),
        col("il.Quantity").alias("quantity"),
        (col("il.UnitPrice") * col("il.Quantity")).alias("total_amount")
    )

fact_sales.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.fact_sales")

print(f"✅ fact_sales: {spark.table(f'{catalog_name}.gold.fact_sales').count()} rows")

In [0]:
agg = spark.table(f"{catalog_name}.gold.fact_sales") \
    .groupBy("customer_key") \
    .agg(
        countDistinct("invoice_id").alias("total_invoices"),
        spark_sum("total_amount").alias("total_spent"),
        spark_sum("quantity").alias("total_items"),
        spark_max("invoice_date").alias("last_purchase_date"),
        spark_min("invoice_date").alias("first_purchase_date")
    )

agg.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.fact_sales_customer_agg")

print(f"✅ fact_sales_customer_agg: {spark.table(f'{catalog_name}.gold.fact_sales_customer_agg').count()} rows")

In [0]:
print("=" * 50)
print("GOLD LAYER SUMMARY")
print("=" * 50)
gold_tables = [
    "dim_artist", "dim_album", "dim_genre", "dim_media_type",
    "dim_employee", "dim_track", "dim_customer",
    "fact_sales", "fact_sales_customer_agg"
]
for tbl in gold_tables:
    cnt = spark.table(f"{catalog_name}.gold.{tbl}").count()
    print(f"  ✅ {tbl}: {cnt} rows")
print("=" * 50)